<a href="https://colab.research.google.com/github/trang1981/ELAPS/blob/main/VIB_LIGHTGBBM_ROS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# LIGHTGBM + ROS PERFORMANCE LADDER — VIB ELAPS
#
# L0: Full cohort
# L1: Remove COUNT_CA_ACCT
# L2: Remove account-feature group
# L3: Placebo-cutoff datasets
# L4: FLDC-60D
#
# Uses the same LightGBM configuration and preprocessing
# as the original LightGBM five-sampling experiment.
# ============================================================

!pip install -q lightgbm imbalanced-learn openpyxl

import os
import glob
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from google.colab import drive

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

from imblearn.pipeline import Pipeline
from imblearn.over_sampling import RandomOverSampler

from lightgbm import LGBMClassifier

warnings.filterwarnings("ignore")


# ============================================================
# 1. MOUNT GOOGLE DRIVE
# ============================================================

drive.mount(
    "/content/drive",
    force_remount=False
)

MYDRIVE_ROOT = "/content/drive/MyDrive"


# ============================================================
# 2. FIND FILE BY EXACT NAME
# ============================================================

def find_file_by_name(
    root,
    filename
):
    matches = glob.glob(
        os.path.join(
            root,
            "**",
            filename
        ),
        recursive=True
    )

    matches = [
        path
        for path in matches
        if os.path.isfile(path)
    ]

    if len(matches) == 0:
        raise FileNotFoundError(
            f"Không tìm thấy file '{filename}' "
            f"trong {root}"
        )

    if len(matches) > 1:
        print(
            f"\nTìm thấy {len(matches)} file "
            f"có tên {filename}:"
        )

        for i, path in enumerate(
            matches,
            start=1
        ):
            print(f"{i}. {path}")

        print(
            "\nCode sẽ sử dụng file đầu tiên."
        )

    return matches[0]


# ============================================================
# 3. INPUT PATHS
# ============================================================

# L0–L2: đúng tên file trong code LightGBM trước
FULL_COHORT_FILE = find_file_by_name(
    MYDRIVE_ROOT,
    "final_dataset_no_auto_job.csv"
)

# L4: đúng dataset FLDC-60D đã sử dụng
FLDC_FILE = os.path.join(
    MYDRIVE_ROOT,
    "Fintect",
    "final_landmark_60d",
    "final_dataset_landmark_60d_no_auto_job.csv"
)

# Kết quả
OUTPUT_DIR = os.path.join(
    MYDRIVE_ROOT,
    "Fintect",
    "lightgbm_ros_performance_ladder"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

print("\nFull-cohort dataset:")
print(FULL_COHORT_FILE)

print("\nFLDC-60D dataset:")
print(FLDC_FILE)

if not os.path.isfile(FLDC_FILE):
    raise FileNotFoundError(
        f"Không tìm thấy FLDC dataset:\n{FLDC_FILE}"
    )


# ============================================================
# 4. PLACEBO FILES FOR L3
# ============================================================
# Code tìm các CSV có "placebo" trong tên.
# Nếu tên file khác, điền thủ công vào PLACEBO_FILES.

PLACEBO_FILES = glob.glob(
    os.path.join(
        MYDRIVE_ROOT,
        "Fintect",
        "**",
        "*placebo*.csv"
    ),
    recursive=True
)

PLACEBO_FILES = sorted([
    path
    for path in PLACEBO_FILES
    if os.path.isfile(path)
])

print("\nPlacebo files found:")

if PLACEBO_FILES:
    for path in PLACEBO_FILES:
        print("-", path)
else:
    print(
        "Không tìm thấy file placebo. "
        "L3 sẽ chưa được chạy."
    )

# Ví dụ điền thủ công nếu cần:
#
# PLACEBO_FILES = [
#     "/content/drive/MyDrive/Fintect/.../placebo_seed_1.csv",
#     "/content/drive/MyDrive/Fintect/.../placebo_seed_2.csv",
#     "/content/drive/MyDrive/Fintect/.../placebo_seed_3.csv",
#     "/content/drive/MyDrive/Fintect/.../placebo_seed_4.csv",
#     "/content/drive/MyDrive/Fintect/.../placebo_seed_5.csv"
# ]


# ============================================================
# 5. GLOBAL CONFIGURATION
# ============================================================

FULL_TARGET = "COUNT_CREDITCARD"
FLDC_TARGET = "TARGET_60D"
ID_COL = "CUSTOMER_NUMBER"

TEST_SIZE = 0.20
RANDOM_STATE = 42

# L1
L1_DROP_FEATURES = [
    "COUNT_CA_ACCT"
]

# L2: account-feature group used in the ablation
L2_DROP_FEATURES = [
    "COUNT_CA_ACCT",
    "AVG_CA_BALANCE",
    "COUNT_TD_ACCT",
    "AVG_TD_BALANCE"
]

# Không đưa các biến target/date/helper vào mô hình
NON_FEATURE_COLUMNS = [
    "CUSTOMER_NUMBER",

    "COUNT_CREDITCARD",
    "TARGET_60D",
    "TARGET",
    "RAW_TARGET",

    "OPEN_CARD_DATE",
    "CARD_OPEN_DATE",
    "FIRST_CARD_DATE",

    "CLIENT_CREATE_DATE",
    "RELATIONSHIP_START_DATE",
    "FEATURE_CUTOFF_DATE",
    "LANDMARK_DATE",

    "EARLY_CARD_60D",
    "HAS_60D_FOLLOWUP",
    "TENURE_AT_CUTOFF",
    "TENURE_DAYS_AT_TARGET"
]


# ============================================================
# 6. EXACT LIGHTGBM CONFIGURATION FROM YOUR CODE
# ============================================================

def create_lightgbm_model():

    return LGBMClassifier(
        n_estimators=500,
        learning_rate=0.05,
        num_leaves=31,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    )


# ============================================================
# 7. LOAD DATA
# ============================================================

def load_dataset(
    file_path
):
    dataframe = pd.read_csv(
        file_path,
        low_memory=False
    )

    dataframe.columns = (
        dataframe.columns
        .astype(str)
        .str.strip()
        .str.upper()
    )

    return dataframe


# ============================================================
# 8. PREPARE X AND Y
# ============================================================

def prepare_xy(
    dataframe,
    target_col,
    drop_features=None
):
    if drop_features is None:
        drop_features = []

    dataframe = dataframe.copy()

    if target_col not in dataframe.columns:
        raise ValueError(
            f"Không tìm thấy target {target_col}.\n"
            f"Các cột hiện có:\n"
            f"{dataframe.columns.tolist()}"
        )

    dataframe[target_col] = pd.to_numeric(
        dataframe[target_col],
        errors="coerce"
    )

    dataframe = dataframe.dropna(
        subset=[target_col]
    ).copy()

    dataframe = dataframe.loc[
        dataframe[target_col].isin([0, 1])
    ].copy()

    y = dataframe[target_col].astype(int)

    columns_to_drop = set(
        NON_FEATURE_COLUMNS
        + drop_features
    )

    feature_columns = [
        col
        for col in dataframe.columns
        if col not in columns_to_drop
    ]

    X = dataframe[
        feature_columns
    ].copy()

    # Giống code LightGBM gốc:
    # object được chuyển sang string
    for col in X.columns:
        if (
            X[col].dtype == "object"
            or pd.api.types.is_string_dtype(
                X[col]
            )
        ):
            X[col] = X[col].astype(str)

    return X, y


# ============================================================
# 9. PREPROCESSOR — SAME AS ORIGINAL CODE
# ============================================================

def create_preprocessor(
    X
):
    numeric_columns = (
        X.select_dtypes(
            include=[
                "number"
            ]
        )
        .columns
        .tolist()
    )

    categorical_columns = (
        X.select_dtypes(
            include=[
                "object",
                "string",
                "category"
            ]
        )
        .columns
        .tolist()
    )

    numeric_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median"
                )
            )
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="most_frequent"
                )
            ),
            (
                "encoder",
                OneHotEncoder(
                    handle_unknown="ignore"
                )
            )
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "num",
                numeric_pipeline,
                numeric_columns
            ),
            (
                "cat",
                categorical_pipeline,
                categorical_columns
            )
        ],
        remainder="drop"
    )

    return preprocessor


# ============================================================
# 10. METRIC: LIFT AT TOP DECILE
# ============================================================

def calculate_lift_at_d1(
    y_true,
    y_prob
):
    ranking = pd.DataFrame({
        "Y_TRUE": np.asarray(y_true),
        "Y_PROB": np.asarray(y_prob)
    })

    ranking = ranking.sort_values(
        "Y_PROB",
        ascending=False
    )

    base_rate = ranking[
        "Y_TRUE"
    ].mean()

    top_n = max(
        1,
        int(
            np.ceil(
                len(ranking) * 0.10
            )
        )
    )

    precision_d1 = (
        ranking
        .head(top_n)["Y_TRUE"]
        .mean()
    )

    if base_rate == 0:
        return np.nan

    return precision_d1 / base_rate


# ============================================================
# 11. TRAIN ONE LADDER CONFIGURATION
# ============================================================

def run_lightgbm_ros(
    dataframe,
    target_col,
    level,
    configuration,
    excluded_features=None
):
    X, y = prepare_xy(
        dataframe=dataframe,
        target_col=target_col,
        drop_features=excluded_features
    )

    X_train, X_test, y_train, y_test = (
        train_test_split(
            X,
            y,
            test_size=TEST_SIZE,
            random_state=RANDOM_STATE,
            stratify=y
        )
    )

    preprocessor = create_preprocessor(
        X_train
    )

    pipeline = Pipeline(
        steps=[
            (
                "preprocess",
                preprocessor
            ),
            (
                "sampler",
                RandomOverSampler(
                    random_state=42
                )
            ),
            (
                "model",
                create_lightgbm_model()
            )
        ]
    )

    pipeline.fit(
        X_train,
        y_train
    )

    y_prob = pipeline.predict_proba(
        X_test
    )[:, 1]

    auc = roc_auc_score(
        y_test,
        y_prob
    )

    lift_d1 = calculate_lift_at_d1(
        y_test,
        y_prob
    )

    result = {
        "Level": level,
        "Configuration": configuration,
        "N": len(y),
        "No. features": X.shape[1],
        "Base rate": y.mean(),
        "AUC": auc,
        "Lift@D1": lift_d1
    }

    print(
        f"{level}: "
        f"AUC={auc:.4f}, "
        f"Lift@D1={lift_d1:.4f}, "
        f"Base rate={y.mean():.4%}"
    )

    return result


# ============================================================
# 12. LOAD FULL-COHORT DATA
# ============================================================

full_df = load_dataset(
    FULL_COHORT_FILE
)

print("\nFull cohort shape:", full_df.shape)
print(
    "Full cohort base rate:",
    full_df[FULL_TARGET].mean()
)


# ============================================================
# 13. RUN L0, L1, L2
# ============================================================

results = []

print("\nRunning L0...")

results.append(
    run_lightgbm_ros(
        dataframe=full_df,
        target_col=FULL_TARGET,
        level="L0",
        configuration=(
            "Full cohort, deployed ELAPS"
        ),
        excluded_features=[]
    )
)

print("\nRunning L1...")

results.append(
    run_lightgbm_ros(
        dataframe=full_df,
        target_col=FULL_TARGET,
        level="L1",
        configuration=(
            "L0 without COUNT_CA_ACCT"
        ),
        excluded_features=L1_DROP_FEATURES
    )
)

print("\nRunning L2...")

results.append(
    run_lightgbm_ros(
        dataframe=full_df,
        target_col=FULL_TARGET,
        level="L2",
        configuration=(
            "L0 without account-feature group"
        ),
        excluded_features=L2_DROP_FEATURES
    )
)


# ============================================================
# 14. RUN L3 — PLACEBO REPLICATIONS
# ============================================================

l3_results = []

if PLACEBO_FILES:

    print(
        f"\nRunning L3 on "
        f"{len(PLACEBO_FILES)} placebo files..."
    )

    for i, file_path in enumerate(
        PLACEBO_FILES,
        start=1
    ):
        print(
            f"\nPlacebo replication {i}:"
        )
        print(file_path)

        placebo_df = load_dataset(
            file_path
        )

        result = run_lightgbm_ros(
            dataframe=placebo_df,
            target_col=FULL_TARGET,
            level=f"L3_{i}",
            configuration=(
                f"Placebo cutoff replication {i}"
            ),
            excluded_features=[]
        )

        result["Source file"] = file_path

        l3_results.append(result)

    l3_auc = np.array([
        item["AUC"]
        for item in l3_results
    ])

    l3_lift = np.array([
        item["Lift@D1"]
        for item in l3_results
    ])

    l3_base_rate = np.array([
        item["Base rate"]
        for item in l3_results
    ])

    results.append({
        "Level": "L3",
        "Configuration": (
            "Placebo cutoffs, "
            "window asymmetry equalized"
        ),
        "N": int(
            np.mean([
                item["N"]
                for item in l3_results
            ])
        ),
        "No. features": int(
            np.mean([
                item["No. features"]
                for item in l3_results
            ])
        ),
        "Base rate": float(
            np.mean(l3_base_rate)
        ),
        "AUC": float(
            np.mean(l3_auc)
        ),
        "AUC SD": float(
            np.std(
                l3_auc,
                ddof=1
            )
            if len(l3_auc) > 1
            else 0
        ),
        "Lift@D1": float(
            np.mean(l3_lift)
        ),
        "Lift@D1 SD": float(
            np.std(
                l3_lift,
                ddof=1
            )
            if len(l3_lift) > 1
            else 0
        )
    })

else:
    print(
        "\nL3 chưa chạy vì không tìm thấy "
        "placebo CSV files."
    )


# ============================================================
# 15. RUN L4 — FLDC-60D
# ============================================================

print("\nRunning L4...")

fldc_df = load_dataset(
    FLDC_FILE
)

results.append(
    run_lightgbm_ros(
        dataframe=fldc_df,
        target_col=FLDC_TARGET,
        level="L4",
        configuration=(
            "FLDC: fixed 60-day window, "
            "forward-looking target"
        ),
        excluded_features=[]
    )
)


# ============================================================
# 16. CREATE FINAL TABLE
# ============================================================

result_df = pd.DataFrame(
    results
)

if "AUC SD" not in result_df.columns:
    result_df["AUC SD"] = np.nan

if "Lift@D1 SD" not in result_df.columns:
    result_df["Lift@D1 SD"] = np.nan

level_order = [
    "L0",
    "L1",
    "L2",
    "L3",
    "L4"
]

result_df["Level"] = pd.Categorical(
    result_df["Level"],
    categories=level_order,
    ordered=True
)

result_df = (
    result_df
    .sort_values("Level")
    .reset_index(drop=True)
)

inflation_channels = {
    "L0": (
        "Feature-level post-adoption leakage"
    ),
    "L1": (
        "+ dominant single feature"
    ),
    "L2": (
        "+ account-relationship group"
    ),
    "L3": (
        "+ window-geometry asymmetry "
        "(approximately)"
    ),
    "L4": (
        "+ window geometry (by construction) "
        "+ onboarding artifact"
    )
}

result_df[
    "Inflation channels excluded"
] = (
    result_df["Level"]
    .astype(str)
    .map(inflation_channels)
)

result_df[
    "Base rate formatted"
] = (
    result_df["Base rate"] * 100
).map(
    lambda value: f"{value:.2f}%"
)

result_df[
    "LightGBM+ROS AUC"
] = result_df.apply(
    lambda row: (
        f"{row['AUC']:.3f} ± "
        f"{row['AUC SD']:.3f}"
        if (
            row["Level"] == "L3"
            and pd.notna(row["AUC SD"])
        )
        else f"{row['AUC']:.3f}"
    ),
    axis=1
)

result_df[
    "LightGBM+ROS Lift@D1"
] = result_df.apply(
    lambda row: (
        f"{row['Lift@D1']:.2f} ± "
        f"{row['Lift@D1 SD']:.2f}"
        if (
            row["Level"] == "L3"
            and pd.notna(
                row["Lift@D1 SD"]
            )
        )
        else f"{row['Lift@D1']:.2f}"
    ),
    axis=1
)

publication_table = result_df[
    [
        "Level",
        "Configuration",
        "Inflation channels excluded",
        "Base rate formatted",
        "LightGBM+ROS AUC",
        "LightGBM+ROS Lift@D1"
    ]
].copy()

publication_table.columns = [
    "Level",
    "Configuration",
    "Inflation channels excluded",
    "Base rate",
    "LightGBM+ROS AUC",
    "LightGBM+ROS Lift@D1"
]


# ============================================================
# 17. PRINT RESULTS
# ============================================================

print("\n" + "=" * 140)
print("LIGHTGBM + ROS PERFORMANCE LADDER")
print("=" * 140)

print(
    publication_table.to_string(
        index=False
    )
)


# ============================================================
# 18. L2–L4 RANGE
# ============================================================

audited_results = result_df.loc[
    result_df["Level"].isin(
        ["L2", "L3", "L4"]
    )
]

if len(audited_results) == 3:

    auc_min = audited_results[
        "AUC"
    ].min()

    auc_max = audited_results[
        "AUC"
    ].max()

    print("\nL2–L4 AUC range:")
    print(
        f"{auc_min:.3f}–{auc_max:.3f}"
    )

    print("\nSentence for manuscript:")

    print(
        "The same ordering holds under "
        "LightGBM+ROS, whose L2–L4 "
        f"configurations fall within "
        f"{auc_min:.3f}–{auc_max:.3f}, "
        "confirming that the observed "
        "convergence is model-agnostic "
        "rather than an XGBoost-specific "
        "artifact."
    )

else:
    print(
        "\nChưa tính được khoảng L2–L4 "
        "vì chưa có kết quả L3."
    )


# ============================================================
# 19. SAVE FILES
# ============================================================

raw_csv = os.path.join(
    OUTPUT_DIR,
    "lightgbm_ros_ladder_raw.csv"
)

publication_csv = os.path.join(
    OUTPUT_DIR,
    "lightgbm_ros_ladder_publication.csv"
)

excel_file = os.path.join(
    OUTPUT_DIR,
    "lightgbm_ros_ladder_results.xlsx"
)

result_df.to_csv(
    raw_csv,
    index=False,
    encoding="utf-8-sig"
)

publication_table.to_csv(
    publication_csv,
    index=False,
    encoding="utf-8-sig"
)

with pd.ExcelWriter(
    excel_file,
    engine="openpyxl"
) as writer:

    publication_table.to_excel(
        writer,
        sheet_name="Publication table",
        index=False
    )

    result_df.to_excel(
        writer,
        sheet_name="Raw results",
        index=False
    )

    if l3_results:
        pd.DataFrame(
            l3_results
        ).to_excel(
            writer,
            sheet_name="L3 placebo runs",
            index=False
        )

print("\nSaved:")
print(raw_csv)
print(publication_csv)
print(excel_file)

Mounted at /content/drive

Full-cohort dataset:
/content/drive/MyDrive/ELAPSPLACEBO/final_dataset_no_auto_job.csv

FLDC-60D dataset:
/content/drive/MyDrive/Fintect/final_landmark_60d/final_dataset_landmark_60d_no_auto_job.csv

Placebo files found:
Không tìm thấy file placebo. L3 sẽ chưa được chạy.

Full cohort shape: (127460, 22)
Full cohort base rate: 0.1546681311784089

Running L0...
L0: AUC=0.9507, Lift@D1=5.0529, Base rate=15.4668%

Running L1...
L1: AUC=0.9504, Lift@D1=5.0504, Base rate=15.4668%

Running L2...
L2: AUC=0.8597, Lift@D1=4.0540, Base rate=15.4668%

L3 chưa chạy vì không tìm thấy placebo CSV files.

Running L4...
L4: AUC=0.8994, Lift@D1=6.2142, Base rate=4.6329%

LIGHTGBM + ROS PERFORMANCE LADDER
Level                                     Configuration                               Inflation channels excluded Base rate LightGBM+ROS AUC LightGBM+ROS Lift@D1
   L0                       Full cohort, deployed ELAPS                       Feature-level post-adoption leakage  

PLACEBO

In [3]:
# ============================================================
# L3 PLACEBO CUTOFF — LIGHTGBM + ROS
# EXACT FIVE FINAL PLACEBO DATASETS
#
# Input:
#   final_dataset_placebo_seed_1_no_auto_job.csv
#   ...
#   final_dataset_placebo_seed_5_no_auto_job.csv
#
# Output:
#   - Result for each placebo seed
#   - Mean ± SD for AUC and Lift@D1
#   - CSV and Excel files
#
# GOOGLE COLAB — COPY AND RUN
# ============================================================

!pip install -q lightgbm imbalanced-learn openpyxl

import os
import gc
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from google.colab import drive

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

from imblearn.pipeline import Pipeline
from imblearn.over_sampling import RandomOverSampler

from lightgbm import LGBMClassifier

warnings.filterwarnings("ignore")


# ============================================================
# 1. MOUNT GOOGLE DRIVE
# ============================================================

drive.mount(
    "/content/drive",
    force_remount=False
)


# ============================================================
# 2. EXACT INPUT AND OUTPUT PATHS
# ============================================================

BASE_DIR = Path(
    "/content/drive/MyDrive/ELAPSPLACEBO"
)

INPUT_DIR = (
    BASE_DIR
    / "final_placebo_datasets"
)

OUTPUT_DIR = (
    BASE_DIR
    / "lightgbm_ros_placebo_results"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

SEEDS = [1, 2, 3, 4, 5]

INPUT_PATTERN = (
    "final_dataset_placebo_seed_"
    "{seed}_no_auto_job.csv"
)

ID_COL = "CUSTOMER_NUMBER"
TARGET_COL = "COUNT_CREDITCARD"

TEST_SIZE = 0.20
SPLIT_RANDOM_STATE = 42
ROS_RANDOM_STATE = 42
MODEL_RANDOM_STATE = 42


# ============================================================
# 3. COLUMNS EXCLUDED FROM MODELING
# ============================================================

EXCLUDE_COLS = [
    ID_COL,
    TARGET_COL,

    # Target-related columns
    "TARGET",
    "RAW_TARGET",
    "COUNT_CREDITCARD",
    "MAX_CARD",
    "TOTAL_CARD",
    "FIRST_CARD_MONTH",

    # Dates and cutoff construction variables
    "TARGET_MONTH",
    "CLIENT_CREATE_DATE",
    "RELATIONSHIP_START_DATE",
    "ACTUAL_D_STAR",
    "AVAILABLE_DAYS",
    "ASSIGNED_CUTOFF_DAYS",
    "PLACEBO_CUTOFF_DATE",
    "CUTOFF_TYPE",
    "ELIGIBLE_DSTAR_COUNT",
    "SAMPLING_STATUS",
    "PLACEBO_SEED",
    "TENURE_AT_CUTOFF",
    "TENURE_DAYS_AT_TARGET"
]


# ============================================================
# 4. EXACT LIGHTGBM CONFIGURATION USED PREVIOUSLY
# ============================================================

def create_lightgbm_model():

    return LGBMClassifier(
        n_estimators=500,
        learning_rate=0.05,
        num_leaves=31,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary",
        random_state=MODEL_RANDOM_STATE,
        n_jobs=-1,
        verbosity=-1
    )


# ============================================================
# 5. VERIFY ALL FIVE FILES
# ============================================================

input_files = {
    seed: (
        INPUT_DIR
        / INPUT_PATTERN.format(seed=seed)
    )
    for seed in SEEDS
}

missing_files = [
    str(file_path)
    for file_path in input_files.values()
    if not file_path.is_file()
]

if missing_files:
    raise FileNotFoundError(
        "Không tìm thấy các file placebo sau:\n"
        + "\n".join(missing_files)
    )

print("=" * 90)
print("PLACEBO INPUT FILES")
print("=" * 90)

for seed, file_path in input_files.items():
    print(f"Seed {seed}: {file_path}")


# ============================================================
# 6. LOAD ONE PLACEBO DATASET
# ============================================================

def load_placebo_dataset(
    seed: int
):

    file_path = input_files[seed]

    df = pd.read_csv(
        file_path,
        low_memory=False
    )

    df.columns = (
        df.columns
        .astype(str)
        .str.strip()
        .str.upper()
    )

    if TARGET_COL not in df.columns:
        raise ValueError(
            f"Seed {seed} thiếu target {TARGET_COL}.\n"
            f"Các cột hiện có:\n"
            f"{df.columns.tolist()}"
        )

    if ID_COL not in df.columns:
        raise ValueError(
            f"Seed {seed} thiếu ID {ID_COL}."
        )

    # Standardize target
    df[TARGET_COL] = pd.to_numeric(
        df[TARGET_COL],
        errors="coerce"
    )

    df = df.dropna(
        subset=[TARGET_COL]
    ).copy()

    df = df.loc[
        df[TARGET_COL].isin([0, 1])
    ].copy()

    df[TARGET_COL] = (
        df[TARGET_COL]
        .astype(np.int8)
    )

    duplicate_count = int(
        df[ID_COL]
        .duplicated()
        .sum()
    )

    if duplicate_count > 0:
        raise ValueError(
            f"Seed {seed} có {duplicate_count} "
            f"{ID_COL} bị trùng."
        )

    return df


# ============================================================
# 7. PREPARE X AND Y
# ============================================================

def prepare_xy(
    df: pd.DataFrame
):

    feature_columns = [
        column
        for column in df.columns
        if column not in EXCLUDE_COLS
    ]

    if not feature_columns:
        raise ValueError(
            "Không còn feature nào để huấn luyện."
        )

    X = df[
        feature_columns
    ].copy()

    y = df[
        TARGET_COL
    ].copy()

    # Same convention as the earlier LightGBM experiment:
    # object columns are treated as categorical strings.
    for column in X.columns:

        if (
            pd.api.types.is_object_dtype(
                X[column]
            )
            or
            pd.api.types.is_string_dtype(
                X[column]
            )
            or
            isinstance(
                X[column].dtype,
                pd.CategoricalDtype
            )
        ):
            X[column] = (
                X[column]
                .astype("string")
            )

    return X, y, feature_columns


# ============================================================
# 8. CREATE PREPROCESSOR
# ============================================================

def create_preprocessor(
    X_train: pd.DataFrame
):

    numeric_columns = (
        X_train
        .select_dtypes(
            include=["number"]
        )
        .columns
        .tolist()
    )

    categorical_columns = (
        X_train
        .select_dtypes(
            include=[
                "object",
                "string",
                "category"
            ]
        )
        .columns
        .tolist()
    )

    transformers = []

    if numeric_columns:
        transformers.append(
            (
                "num",
                SimpleImputer(
                    strategy="median"
                ),
                numeric_columns
            )
        )

    if categorical_columns:
        transformers.append(
            (
                "cat",
                Pipeline(
                    steps=[
                        (
                            "imputer",
                            SimpleImputer(
                                strategy="most_frequent"
                            )
                        ),
                        (
                            "encoder",
                            OneHotEncoder(
                                handle_unknown="ignore"
                            )
                        )
                    ]
                ),
                categorical_columns
            )
        )

    if not transformers:
        raise ValueError(
            "Không xác định được cột số hoặc cột phân loại."
        )

    return ColumnTransformer(
        transformers=transformers,
        remainder="drop"
    )


# ============================================================
# 9. LIFT AT TOP DECILE
# ============================================================

def calculate_lift_at_d1(
    y_true,
    y_probability
):

    ranking = pd.DataFrame({
        "Y_TRUE": np.asarray(y_true),
        "Y_PROB": np.asarray(y_probability)
    })

    ranking = (
        ranking
        .sort_values(
            "Y_PROB",
            ascending=False
        )
        .reset_index(drop=True)
    )

    base_rate = float(
        ranking["Y_TRUE"].mean()
    )

    # Top 10% of the holdout
    top_n = max(
        1,
        int(
            np.ceil(
                len(ranking) * 0.10
            )
        )
    )

    precision_at_d1 = float(
        ranking
        .head(top_n)["Y_TRUE"]
        .mean()
    )

    lift_at_d1 = (
        precision_at_d1 / base_rate
        if base_rate > 0
        else np.nan
    )

    return {
        "Top decile n": top_n,
        "Base rate": base_rate,
        "Precision@D1": precision_at_d1,
        "Lift@D1": lift_at_d1
    }


# ============================================================
# 10. RUN ONE PLACEBO SEED
# ============================================================

def run_one_seed(
    seed: int
):

    print("\n" + "=" * 90)
    print(f"LIGHTGBM + ROS — PLACEBO SEED {seed}")
    print("=" * 90)

    df = load_placebo_dataset(
        seed
    )

    X, y, feature_columns = prepare_xy(
        df
    )

    print("Dataset shape :", df.shape)
    print("Customers     :", len(y))
    print("Features      :", len(feature_columns))
    print("Positive cases:", int(y.sum()))
    print(
        "Base rate     :",
        f"{y.mean():.4%}"
    )

    print("\nFeatures:")
    print(feature_columns)

    # Same split seed for all placebo datasets.
    # Variation across runs therefore comes from the placebo
    # feature construction, not from a different holdout split.
    X_train, X_test, y_train, y_test = (
        train_test_split(
            X,
            y,
            test_size=TEST_SIZE,
            random_state=SPLIT_RANDOM_STATE,
            stratify=y
        )
    )

    preprocessor = create_preprocessor(
        X_train
    )

    model_pipeline = Pipeline(
        steps=[
            (
                "preprocess",
                preprocessor
            ),
            (
                "sampler",
                RandomOverSampler(
                    random_state=ROS_RANDOM_STATE
                )
            ),
            (
                "model",
                create_lightgbm_model()
            )
        ]
    )

    model_pipeline.fit(
        X_train,
        y_train
    )

    y_probability = (
        model_pipeline
        .predict_proba(
            X_test
        )[:, 1]
    )

    test_auc = float(
        roc_auc_score(
            y_test,
            y_probability
        )
    )

    ranking_metrics = (
        calculate_lift_at_d1(
            y_test,
            y_probability
        )
    )

    result = {
        "Seed": seed,
        "Rows": int(len(df)),
        "Features": int(len(feature_columns)),
        "Positive cases": int(y.sum()),
        "Base rate": float(y.mean()),
        "Train rows": int(len(y_train)),
        "Test rows": int(len(y_test)),
        "Test positives": int(y_test.sum()),
        "AUC": test_auc,
        "Precision@D1": (
            ranking_metrics[
                "Precision@D1"
            ]
        ),
        "Lift@D1": (
            ranking_metrics[
                "Lift@D1"
            ]
        ),
        "Input file": str(
            input_files[seed]
        )
    }

    print("\nResult:")
    print(f"AUC         : {test_auc:.6f}")
    print(
        "Precision@D1:",
        f"{result['Precision@D1']:.6f}"
    )
    print(
        "Lift@D1     :",
        f"{result['Lift@D1']:.6f}"
    )

    del (
        df,
        X,
        y,
        X_train,
        X_test,
        y_train,
        y_test,
        model_pipeline,
        y_probability
    )

    gc.collect()

    return result


# ============================================================
# 11. RUN ALL FIVE PLACEBO SEEDS
# ============================================================

seed_results = []

for seed in SEEDS:

    seed_result = run_one_seed(
        seed
    )

    seed_results.append(
        seed_result
    )

result_df = pd.DataFrame(
    seed_results
)


# ============================================================
# 12. CALCULATE L3 MEAN ± SAMPLE SD
# ============================================================

auc_mean = float(
    result_df["AUC"].mean()
)

auc_sd = float(
    result_df["AUC"].std(ddof=1)
)

lift_mean = float(
    result_df["Lift@D1"].mean()
)

lift_sd = float(
    result_df["Lift@D1"].std(ddof=1)
)

precision_d1_mean = float(
    result_df["Precision@D1"].mean()
)

precision_d1_sd = float(
    result_df["Precision@D1"].std(ddof=1)
)

base_rate_mean = float(
    result_df["Base rate"].mean()
)

summary_df = pd.DataFrame([
    {
        "Level": "L3",
        "Configuration": (
            "Placebo cutoffs, "
            "window asymmetry equalized"
        ),
        "Model": "LightGBM + ROS",
        "Replications": len(result_df),
        "Mean rows": result_df[
            "Rows"
        ].mean(),
        "Mean base rate": base_rate_mean,
        "AUC mean": auc_mean,
        "AUC SD": auc_sd,
        "Precision@D1 mean": (
            precision_d1_mean
        ),
        "Precision@D1 SD": (
            precision_d1_sd
        ),
        "Lift@D1 mean": lift_mean,
        "Lift@D1 SD": lift_sd
    }
])


# ============================================================
# 13. PRINT FINAL RESULTS
# ============================================================

display_columns = [
    "Seed",
    "Rows",
    "Features",
    "Base rate",
    "AUC",
    "Precision@D1",
    "Lift@D1"
]

print("\n" + "=" * 110)
print("LIGHTGBM + ROS — FIVE PLACEBO SEEDS")
print("=" * 110)

print(
    result_df[
        display_columns
    ].to_string(
        index=False,
        formatters={
            "Base rate": (
                lambda value:
                f"{value * 100:.2f}%"
            ),
            "AUC": (
                lambda value:
                f"{value:.6f}"
            ),
            "Precision@D1": (
                lambda value:
                f"{value:.6f}"
            ),
            "Lift@D1": (
                lambda value:
                f"{value:.6f}"
            )
        }
    )
)

print("\n" + "=" * 110)
print("L3 SUMMARY FOR PERFORMANCE LADDER")
print("=" * 110)

print(
    f"Base rate : "
    f"{base_rate_mean * 100:.2f}%"
)

print(
    f"AUC       : "
    f"{auc_mean:.3f} ± {auc_sd:.3f}"
)

print(
    f"Lift@D1   : "
    f"{lift_mean:.2f} ± {lift_sd:.2f}"
)

print("\nTable row:")

print(
    "L3 | Placebo cutoffs, window asymmetry "
    "equalized | "
    f"{base_rate_mean * 100:.2f}% | "
    f"{auc_mean:.3f} ± {auc_sd:.3f} | "
    f"{lift_mean:.2f} ± {lift_sd:.2f}"
)


# ============================================================
# 14. COMPLETE LIGHTGBM LADDER USING EXISTING RESULTS
# ============================================================

ladder_df = pd.DataFrame([
    {
        "Level": "L0",
        "Configuration": (
            "Full cohort, deployed ELAPS"
        ),
        "Base rate": "15.47%",
        "LightGBM+ROS AUC": "0.951",
        "LightGBM+ROS Lift@D1": "5.05"
    },
    {
        "Level": "L1",
        "Configuration": (
            "L0 without COUNT_CA_ACCT"
        ),
        "Base rate": "15.47%",
        "LightGBM+ROS AUC": "0.950",
        "LightGBM+ROS Lift@D1": "5.05"
    },
    {
        "Level": "L2",
        "Configuration": (
            "L0 without account-feature group"
        ),
        "Base rate": "15.47%",
        "LightGBM+ROS AUC": "0.860",
        "LightGBM+ROS Lift@D1": "4.05"
    },
    {
        "Level": "L3",
        "Configuration": (
            "Placebo cutoffs, "
            "window asymmetry equalized"
        ),
        "Base rate": (
            f"{base_rate_mean * 100:.2f}%"
        ),
        "LightGBM+ROS AUC": (
            f"{auc_mean:.3f} ± {auc_sd:.3f}"
        ),
        "LightGBM+ROS Lift@D1": (
            f"{lift_mean:.2f} ± {lift_sd:.2f}"
        )
    },
    {
        "Level": "L4",
        "Configuration": (
            "FLDC: fixed 60-day window, "
            "forward-looking target"
        ),
        "Base rate": "4.63%",
        "LightGBM+ROS AUC": "0.899",
        "LightGBM+ROS Lift@D1": "6.21"
    }
])

print("\n" + "=" * 130)
print("COMPLETE LIGHTGBM + ROS PERFORMANCE LADDER")
print("=" * 130)

print(
    ladder_df.to_string(
        index=False
    )
)


# ============================================================
# 15. L2–L4 AUC RANGE
# ============================================================

l2_auc = 0.860
l3_auc = auc_mean
l4_auc = 0.899

auc_min = min(
    l2_auc,
    l3_auc,
    l4_auc
)

auc_max = max(
    l2_auc,
    l3_auc,
    l4_auc
)

print("\n" + "=" * 90)
print("L2–L4 RANGE")
print("=" * 90)

print(
    f"LightGBM+ROS L2–L4 AUC range: "
    f"{auc_min:.3f}–{auc_max:.3f}"
)

print("\nSuggested manuscript sentence:")

print(
    "The same qualitative pattern was reproduced "
    "under LightGBM+ROS. Across the account-group "
    "ablation, placebo-cutoff, and FLDC configurations, "
    f"AUC ranged from {auc_min:.3f} to {auc_max:.3f}, "
    "indicating that the audited performance pattern "
    "is not specific to XGBoost."
)


# ============================================================
# 16. SAVE OUTPUTS
# ============================================================

seed_output_file = (
    OUTPUT_DIR
    / "lightgbm_ros_l3_placebo_five_seeds.csv"
)

summary_output_file = (
    OUTPUT_DIR
    / "lightgbm_ros_l3_placebo_summary.csv"
)

ladder_output_file = (
    OUTPUT_DIR
    / "lightgbm_ros_complete_ladder.csv"
)

excel_output_file = (
    OUTPUT_DIR
    / "lightgbm_ros_l3_and_complete_ladder.xlsx"
)

result_df.to_csv(
    seed_output_file,
    index=False,
    encoding="utf-8-sig"
)

summary_df.to_csv(
    summary_output_file,
    index=False,
    encoding="utf-8-sig"
)

ladder_df.to_csv(
    ladder_output_file,
    index=False,
    encoding="utf-8-sig"
)

with pd.ExcelWriter(
    excel_output_file,
    engine="openpyxl"
) as writer:

    result_df.to_excel(
        writer,
        sheet_name="L3 five seeds",
        index=False
    )

    summary_df.to_excel(
        writer,
        sheet_name="L3 summary",
        index=False
    )

    ladder_df.to_excel(
        writer,
        sheet_name="Complete ladder",
        index=False
    )

print("\nSaved files:")
print(seed_output_file)
print(summary_output_file)
print(ladder_output_file)
print(excel_output_file)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PLACEBO INPUT FILES
Seed 1: /content/drive/MyDrive/ELAPSPLACEBO/final_placebo_datasets/final_dataset_placebo_seed_1_no_auto_job.csv
Seed 2: /content/drive/MyDrive/ELAPSPLACEBO/final_placebo_datasets/final_dataset_placebo_seed_2_no_auto_job.csv
Seed 3: /content/drive/MyDrive/ELAPSPLACEBO/final_placebo_datasets/final_dataset_placebo_seed_3_no_auto_job.csv
Seed 4: /content/drive/MyDrive/ELAPSPLACEBO/final_placebo_datasets/final_dataset_placebo_seed_4_no_auto_job.csv
Seed 5: /content/drive/MyDrive/ELAPSPLACEBO/final_placebo_datasets/final_dataset_placebo_seed_5_no_auto_job.csv

LIGHTGBM + ROS — PLACEBO SEED 1
Dataset shape : (127460, 22)
Customers     : 127460
Features      : 20
Positive cases: 19714
Base rate     : 15.4668%

Features:
['CLIENT_SEX', 'EB_REGISTER_CHANNEL', 'SMS', 'VERIFY_METHOD', 'AGE', 'LOGIN_PER_ACTIVE_DAY', 'INTEREST_RATE_RATIO', 'TRANS_LV1_MO